<a href="https://colab.research.google.com/github/danimb-git/cardio-disease-ml-uab/blob/main/notebooks/Main_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicting the presence of cardiovascular disease
**Course:** Fundamentals of Machine Learning  
**Project team:** Team 8 - Ariadna Lopez Traver, Daniel Moore Burgos, Alejandro Ucelayeta Jarque  
**Dataset:** [Cardiovascular Disease Dataset (Kaggle)](https://www.kaggle.com/datasets/sulianova/cardiovascular-disease-dataset?resource=download$0)

## 1. Project Objective & Problem Formulation
Cardiovascular diseases (CVDs) are a group of conditions affecting the heart and blood vessels, and are one of the leading causes of mortality globally.

The objective of this project is to build, evaluate, and analyze supervised binary classification models capable of predicting the presence or absence of CVDs drom clinical examination data, patient metrics, and lifestyle habits.

### Dataset features

The dataset consists of 70 000 records of patients data in 12 features, such as age, gender, systolic blood pressure, diastolic blood pressure, etc. The target class "cardio" equals to 1, when patient has cardiovascular desease, and it's 0, if patient is healthy.

There are 3 types of input features:
* Objective: factual information.
* Examination: results of medical examination.
* Subjective: information given by the patient.

| Feature | Variable Type | Variable | Value Type | Description |
| :--- | :--- | :--- | :--- | :--- |
| Age | Objective Feature | `age` | int (days) | Patient's age in days |
| Height | Objective Feature | `height` | int (cm) | Patient's height |
| Weight | Objective Feature | `weight` | float (kg) | Patient's weight |
| Gender | Objective Feature | `gender` | categorical code | 1: women, 2: men |
| Systolic blood pressure | Examination Feature | `ap_hi` | int (mmHg) | Systolic blood pressure |
| Diastolic blood pressure | Examination Feature | `ap_lo` | int (mmHg) | Diastolic blood pressure |
| Cholesterol | Examination Feature | `cholesterol` | ordinal (1, 2, 3) | 1: normal, 2: above normal, 3: well above normal |
| Glucose | Examination Feature | `gluc` | ordinal (1, 2, 3) | 1: normal, 2: above normal, 3: well above normal |
| Smoking | Subjective Feature | `smoke` | binary | Self-reported smoking status |
| Alcohol intake | Subjective Feature | `alco` | binary | Self-reported alcohol intake |
| Physical activity | Subjective Feature | `active` | binary | Self-reported physical activity |
| Cardio (Target) | Target Variable | `cardio` | binary | Presence (1) or absence (0) of cardiovascular disease |

### Methodoogy

To ensure methodological rigor, we follow the OSEMN workflow: Obtain, Scrub, Explore, Model, iNterpret.

To ensure reproducibility across all team members, the dataset is split into training and test sets immediately after basic structural verification. All exploratory data analysis, outlier handling, feature enguneering, scaling, and cross-validation will be performed exclusively on the training set.

As a team we'll work using a shared Google Drive workspace with a single master notebook *Main_Pipeline.ipynb*. Every code block will be documented with analytical takeaways.

## 2. Enviroment setup and reproducibility

In this section we import the main scientific computing and visualization libraries we'll use throughout the notebook, define a global random seed to ensure reproducibility of data splits, and log the exact library versions used in the project.

In [40]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split

In [41]:
# Gemini recomends supresing non-critical warnings to keep the notebook outputs clean.
# import warnings
# warnings.filterwarnings('ignore')

In [42]:
# Definition of gloval random seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [43]:
# Gemini recomends setting default values for consistent visual styling for the
# exploratory data analysis and for report figures
# sns.set_theme(style="whitegrid", context="notebook", palette="muted")
# plt.rcParams["figure.figsize"] = (10, 6)
# plt.rcParams["axes.titlesize"] = 13
# plt.rcParams["axes.labelsize"] = 11
# Display all columns when inspecting DataFrames
# pd.set_option("display.max_columns", None)
# pd.set_option("display.float_format", lambda x: f"{x:.4f}")

In [44]:
# Library versions
print("Python version :", sys.version.split()[0])
print("NumPy version :", np.__version__)
print("Pandas version :", pd.__version__)
print("Scikit-learn version :", sklearn.__version__)
print("Global RANDOM_SEED :", RANDOM_SEED)

Python version : 3.13.18
NumPy version : 2.4.6
Pandas version : 2.2.3
Scikit-learn version : 1.6.1
Global RANDOM_SEED : 42


## 3. Data acquisition and structural audit

Before splitting the data into the training and test sets, we perforn structural checks of the dataset (dimensions, data types, duplicates, etc.).
1. Lead the CSV file (Kaggle files use ";" as delimiter).
2. Dataset dimensions, column names, and data types.
3. Check if the dataset contains missing values (NaN).
4. Remove the `id` column. It's an arbitrary identifier with no predictive meaning (models should not have the id to learn from it).
5. Check for duplicate rows (exact duplicates exluding `id`). Leaving identical records across both the training and test sets would inflate test metrics.

In [45]:
local_path = "cardio_train.csv"
github_url = "https://raw.githubusercontent.com/danimb-git/cardio-disease-ml-uab/main/cardio_train.csv"

file = local_path
if not os.path.exists(file):
    file = github_url
    print("Loading dataset from GitHub Raw URL")
else:
    print("Loading dataset from local path")

df_raw = pd.read_csv(file, sep=';')

print("Raw dataset shape:", df_raw.shape[0], "rows x", df_raw.shape[1], "columns")
display(df_raw.head())

Loading dataset from local path
Raw dataset shape: 70000 rows x 13 columns


,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0
